# 01A — Petrobras 3W sector pack

**Outcome:** translate a deterministic development population of real-well
Petrobras 3W recordings into the same Pack v0.6 metric-level interface used by Telecom.

Each source recording becomes one observation episode. The notebook keeps
`class` and `state` in
`PACK-EVAL`, never in model input. It does not invent manifold topology,
tickets, maintenance events, severity or cross-well causes.


## 1. Setup

The official 3W 2.0.0 directory is expected under Drive. The default
development population selects one deterministic **real WELL** recording
for every available `(well, event class)` pair. This is substantially larger
than the old three-file contract fixture while remaining practical in Colab.


In [ ]:
import configparser
import hashlib
import os
import shutil
import sys
import tempfile
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = Path(os.getenv(
    "ANOMALY_DRIVE_ROOT",
    "/content/drive/MyDrive/anomaly_detection",
))
NOTEBOOK_HOME = Path(os.getenv(
    "ANOMALY_NOTEBOOK_HOME",
    DRIVE_ROOT / "research" / "milestone1",
))
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from milestone1_core import (
    EVAL_SCHEMAS,
    PACK_ENTITY_SCHEMA,
    PACK_EPISODE_SCHEMA,
    PACK_METRIC_SCHEMA,
    PACK_OBSERVATION_SCHEMA,
    SPLIT_SCHEMAS,
    save_pack,
    new_output_directory,
    pack_fingerprint,
    read_json,
    source_file,
)

SOURCE = Path(os.getenv(
    "THREEW_SOURCE_ROOT",
    DRIVE_ROOT / "sources" / "petrobras_3w" / "2.0.0" / "raw" / "3w_dataset_2.0.0",
))
PACK_RUN_ID = os.getenv("THREEW_PACK_RUN_ID", "real_wells_expanded_v0_6_2")
PACK_ROOT = DRIVE_ROOT / "outputs" / "packs" / "petrobras_3w" / PACK_RUN_ID
FILES_PER_WELL_EVENT = int(os.getenv("THREEW_FILES_PER_WELL_EVENT", "1"))
BATCH_ROWS = int(os.getenv("THREEW_BATCH_ROWS", "50000"))
RUN_BUILD = os.getenv("RUN_THREEW_PACK", "1") == "1"

display(pd.Series({
    "source": str(SOURCE),
    "pack_root": str(PACK_ROOT),
    "files_per_well_event": FILES_PER_WELL_EVENT,
    "source_kind": "real wells only",
}, name="value").to_frame())


## 2. Petrobras 3W phrasebook

The standardized names are readable identifiers. Units follow the official
`dataset.ini`. `sampling_mode="recording"` is important: observations are
periodic within a file, but the files do not establish a continuous
reporting obligation between recorded events.


In [ ]:
THREEW_CADENCE_SECONDS = 1
METRIC_COLUMNS = ["native_field", *PACK_METRIC_SCHEMA]

def metric(native, metric_id, kind, unit):
    return (
        native, metric_id, "oil_well", kind, unit,
        "recording", THREEW_CADENCE_SECONDS,
    )


metric_map = pd.DataFrame([
    metric("ABER-CKGL", "gas_lift_choke_opening", "gauge", "percent"),
    metric("ABER-CKP", "production_choke_opening", "gauge", "percent"),
    metric("ESTADO-DHSV", "downhole_safety_valve_state", "discrete_state", "state_code"),
    metric("ESTADO-M1", "production_master_valve_state", "discrete_state", "state_code"),
    metric("ESTADO-M2", "annulus_master_valve_state", "discrete_state", "state_code"),
    metric("ESTADO-PXO", "pig_crossover_valve_state", "discrete_state", "state_code"),
    metric("ESTADO-SDV-GL", "gas_lift_shutdown_valve_state", "discrete_state", "state_code"),
    metric("ESTADO-SDV-P", "production_shutdown_valve_state", "discrete_state", "state_code"),
    metric("ESTADO-W1", "production_wing_valve_state", "discrete_state", "state_code"),
    metric("ESTADO-W2", "annulus_wing_valve_state", "discrete_state", "state_code"),
    metric("ESTADO-XO", "crossover_valve_state", "discrete_state", "state_code"),
    metric("P-ANULAR", "annulus_pressure", "gauge", "Pa"),
    metric("P-JUS-BS", "service_pump_downstream_pressure", "gauge", "Pa"),
    metric("P-JUS-CKGL", "gas_lift_choke_downstream_pressure", "gauge", "Pa"),
    metric("P-JUS-CKP", "production_choke_downstream_pressure", "gauge", "Pa"),
    metric("P-MON-CKGL", "gas_lift_choke_upstream_pressure", "gauge", "Pa"),
    metric("P-MON-CKP", "production_choke_upstream_pressure", "gauge", "Pa"),
    metric("P-MON-SDV-P", "production_shutdown_valve_upstream_pressure", "gauge", "Pa"),
    metric("P-PDG", "downhole_pressure", "gauge", "Pa"),
    metric("PT-P", "production_tube_downstream_pressure", "gauge", "Pa"),
    metric("P-TPT", "tubing_pressure", "gauge", "Pa"),
    metric("QBS", "service_pump_flow_rate", "gauge", "m3/s"),
    metric("QGL", "gas_lift_flow_rate", "gauge", "m3/s"),
    metric("T-JUS-CKP", "production_choke_downstream_temperature", "gauge", "degC"),
    metric("T-MON-CKP", "production_choke_upstream_temperature", "gauge", "degC"),
    metric("T-PDG", "downhole_temperature", "gauge", "degC"),
    metric("T-TPT", "tubing_temperature", "gauge", "degC"),
], columns=METRIC_COLUMNS)

assert len(metric_map) == 27
assert metric_map["metric_id"].is_unique
assert {"class", "state"}.isdisjoint(metric_map["native_field"])
display(metric_map)


## 3. Inventory, development population and whole-well split

Duplicate download variants such as `file (1).parquet` are rejected by the
official 2,228-file count. Real files are those whose names begin `WELL-`.

Selection is deterministic but does not split rows or timestamps. Entire
wells are assigned to calibration, development or holdout. A reproducible
hash search requires normal and event recordings in both evaluation
partitions. Missing individual event types are reported, not fabricated.


In [ ]:
EXPECTED_FILES = 2_228

parser = configparser.ConfigParser(interpolation=None)
parser.read(SOURCE / "dataset.ini", encoding="utf-8")
version = parser.get("VERSION", "DATASET")
event_files = [
    path
    for event_code in range(10)
    for path in sorted((SOURCE / str(event_code)).glob("*.parquet"))
]
missing_directories = [
    str(code) for code in range(10)
    if not (SOURCE / str(code)).is_dir()
]
real_files = [
    path for path in event_files
    if path.name.startswith("WELL-") and " (" not in path.stem
]

inventory = pd.DataFrame({
    "path": real_files,
    "relative_path": [str(path.relative_to(SOURCE)) for path in real_files],
    "entity_id": [path.name.split("_", 1)[0] for path in real_files],
    "event_code": [int(path.parent.name) for path in real_files],
    "source_instance_id": [path.stem for path in real_files],
    "rows": [pq.ParquetFile(path).metadata.num_rows for path in real_files],
})
inventory["selection_key"] = inventory["relative_path"].map(
    lambda value: hashlib.sha256(value.encode()).hexdigest()
)
selected = (
    inventory.sort_values("selection_key")
    .groupby(["entity_id", "event_code"], as_index=False)
    .head(FILES_PER_WELL_EVENT)
    .sort_values(["entity_id", "event_code", "relative_path"])
    .reset_index(drop=True)
)


def available_threew_catalogue(source, selection):
    """Keep metrics with at least one non-null real-well observation."""

    remaining = set(metric_map["native_field"])
    available = set()
    for relative_path in selection["relative_path"]:
        parquet = pq.ParquetFile(Path(source) / relative_path)
        names = parquet.schema_arrow.names
        for field in sorted(remaining.intersection(names)):
            column_index = names.index(field)
            statistics = [
                parquet.metadata.row_group(group).column(column_index).statistics
                for group in range(parquet.metadata.num_row_groups)
            ]
            if all(item is not None and item.null_count is not None for item in statistics):
                has_value = any(
                    item.null_count < parquet.metadata.row_group(group).num_rows
                    for group, item in enumerate(statistics)
                )
            else:
                values = pd.read_parquet(Path(source) / relative_path, columns=[field])
                has_value = values[field].notna().any()
            if has_value:
                available.add(field)
        remaining -= available
        if not remaining:
            break
    return metric_map.loc[metric_map["native_field"].isin(available)].copy()


selected_catalogue = available_threew_catalogue(SOURCE, selected)
unavailable_metrics = sorted(
    set(metric_map["native_field"]) - set(selected_catalogue["native_field"])
)

def make_well_partitions(selection, max_attempts=5_000):
    """Create a reproducible whole-well split with minimum coverage."""

    wells = sorted(selection["entity_id"].unique())
    n_calibration = round(len(wells) * 0.50)
    n_development = round(len(wells) * 0.25)
    n_holdout = len(wells) - n_calibration - n_development
    if min(n_calibration, n_development, n_holdout) < 1:
        raise ValueError("Not enough wells for three partitions")

    for attempt in range(max_attempts):
        ordered_wells = sorted(
            wells,
            key=lambda well: hashlib.sha256(
                f"threew_split_v2|{attempt}|{well}".encode()
            ).hexdigest(),
        )
        partition_by_well = {
            well: (
                "calibration" if index < n_calibration
                else "development" if index < n_calibration + n_development
                else "holdout"
            )
            for index, well in enumerate(ordered_wells)
        }
        candidate = selection.assign(
            partition=selection["entity_id"].map(partition_by_well)
        )
        coverage_is_valid = True
        for partition in ["development", "holdout"]:
            event_codes = set(
                candidate.loc[candidate["partition"].eq(partition), "event_code"]
            )
            if 0 not in event_codes or not any(code != 0 for code in event_codes):
                coverage_is_valid = False
                break
        if coverage_is_valid:
            return partition_by_well, attempt

    raise ValueError(
        "Could not create a whole-well split with normal and event "
        "coverage in development and holdout"
    )


partition_by_well, split_attempt = make_well_partitions(selected)
wells = sorted(partition_by_well)
selected["partition"] = selected["entity_id"].map(partition_by_well)
entity_partitions = pd.DataFrame({
    "entity_id": wells,
    "partition": [partition_by_well[well] for well in wells],
    "split_version": "threew_well_hash_coverage_v2",
})[SPLIT_SCHEMAS["entity_partitions"]]

source_summary = {
    "version": version,
    "event_file_count": len(event_files),
    "real_file_count": len(real_files),
    "selected_file_count": len(selected),
    "selected_wells": selected["entity_id"].nunique(),
    "selected_rows": int(selected["rows"].sum()),
    "metrics_with_observations": len(selected_catalogue),
    "all_null_metrics_excluded": unavailable_metrics,
    "missing_event_directories": missing_directories,
    "accepted_split_attempt": split_attempt,
}
split_coverage = pd.crosstab(selected["partition"], selected["event_code"])
missing_codes = {
    partition: sorted(set(range(10)) - set(group["event_code"]))
    for partition, group in selected.groupby("partition")
}
source_summary["missing_event_codes_by_partition"] = missing_codes
display(pd.Series(source_summary, name="value").to_frame())
display(split_coverage)
display(entity_partitions.groupby("partition").size().rename("wells").to_frame())
display(selected.head(20))

assert version == "2.0.0"
assert len(event_files) == EXPECTED_FILES
assert not missing_directories
assert not selected_catalogue.empty
assert selected["entity_id"].nunique() >= 10
assert {0, 1, 2, 3, 4, 5, 6, 7, 8, 9} <= set(selected["event_code"])
assert set(entity_partitions["partition"]) == {"calibration", "development", "holdout"}
assert entity_partitions.groupby("entity_id")["partition"].nunique().max() == 1
assert (split_coverage.loc[["development", "holdout"], 0] > 0).all()
assert (split_coverage.loc[["development", "holdout"], split_coverage.columns != 0].sum(axis=1) > 0).all()


## 4. Translate 3W observations and labels

`observable_ts` and `impact_ts` are derived from the published `1xx`
transient and `x` steady class phases. Their `label_source` says this
explicitly; they are not independent physical or business-impact timestamps.


In [ ]:
def event_definitions(source):
    """Read event names and transient rules from dataset.ini."""

    metadata_path = Path(source) / "dataset.ini"
    parser = configparser.ConfigParser(interpolation=None)
    if not parser.read(metadata_path, encoding="utf-8"):
        raise FileNotFoundError(f"Missing 3W metadata: {metadata_path}")
    names = [
        item.strip()
        for item in parser.get("EVENTS", "NAMES").replace("\n", "").split(",")
        if item.strip()
    ]
    definitions = {}
    for name in names:
        event_code = parser.getint(name, "LABEL")
        definitions[event_code] = {
            "description": parser.get(name, "DESCRIPTION"),
            "has_transient": parser.getboolean(
                name, "TRANSIENT", fallback=event_code != 0
            ),
        }
    if len(definitions) != len(names):
        raise ValueError("dataset.ini contains duplicate event labels")
    transient_offset = parser.getint("EVENTS", "TRANSIENT_OFFSET", fallback=100)
    return definitions, transient_offset


def contiguous_runs(values):
    """Return (start, end, value) for each run; end is exclusive."""

    values = list(values)
    if not values:
        return []
    rows = []
    start, current = 0, values[0]
    for index, value in enumerate(values[1:], start=1):
        same = (pd.isna(value) and pd.isna(current)) or (
            pd.notna(value) and pd.notna(current) and value == current
        )
        if not same:
            rows.append((start, index, current))
            start, current = index, value
    rows.append((start, len(values), current))
    return rows


def read_threew_file(path):
    """Read one recording and validate its time axis."""

    frame = pd.read_parquet(path)
    if frame.empty:
        raise ValueError(f"Empty 3W recording: {path}")
    if "timestamp" in frame.columns:
        frame = frame.rename(columns={"timestamp": "event_ts"}).reset_index(drop=True)
    elif isinstance(frame.index, pd.DatetimeIndex):
        frame = frame.reset_index().rename(columns={frame.index.name or "index": "event_ts"})
    else:
        raise ValueError(f"No timestamp column or DatetimeIndex in {path}")

    frame["event_ts"] = pd.to_datetime(frame["event_ts"], utc=True, errors="raise")
    if frame["event_ts"].isna().any():
        raise ValueError(f"Missing timestamps in {path}")
    if not frame["event_ts"].is_monotonic_increasing:
        raise ValueError(f"Timestamps are not ordered in {path}")
    if frame["event_ts"].duplicated().any():
        raise ValueError(f"Duplicate timestamps in {path}")
    return frame.reset_index(drop=True)


def threew_truth(frame, relative_path, definitions, transient_offset):
    """Convert 3W class and state labels into evaluation-only intervals."""

    path = Path(relative_path)
    entity_id = path.name.split("_", 1)[0]
    instance_id = path.stem
    event_code = int(path.parent.name)
    missing = {"event_ts", "class", "state"} - set(frame.columns)
    if missing:
        raise ValueError(f"Evaluation labels missing from {relative_path}: {sorted(missing)}")
    if event_code not in definitions:
        raise ValueError(f"Event {event_code} is not described in dataset.ini")

    timestamps = frame["event_ts"].reset_index(drop=True)
    final_end = timestamps.iloc[-1] + pd.Timedelta(seconds=THREEW_CADENCE_SECONDS)
    labels = pd.to_numeric(frame["class"], errors="raise").reset_index(drop=True)
    observed_labels = labels.dropna()
    if observed_labels.mod(1).ne(0).any():
        raise ValueError(f"Non-integer class label in {relative_path}")
    allowed_labels = set(definitions)
    allowed_labels.update(
        transient_offset + code
        for code, details in definitions.items()
        if details["has_transient"]
    )
    unexpected = set(observed_labels.astype(int)) - allowed_labels
    if unexpected:
        raise ValueError(f"Unexpected class labels in {relative_path}: {sorted(unexpected)}")
    if not labels.eq(event_code).any():
        raise ValueError(
            f"{relative_path} does not contain its declared class {event_code}"
        )

    conditions = []
    for start, end, value in contiguous_runs(frame["state"]):
        if pd.isna(value):
            continue
        condition_code = (
            str(int(value))
            if pd.api.types.is_number(value) and float(value).is_integer()
            else str(value)
        )
        conditions.append({
            "entity_id": entity_id,
            "start_ts": timestamps.iloc[start],
            "end_ts": timestamps.iloc[end] if end < len(timestamps) else final_end,
            "condition_code": condition_code,
            "label_source": "3w_state_label",
            "source_instance_id": instance_id,
        })

    event_codes = labels.map(
        lambda value: (
            pd.NA if pd.isna(value) or int(value) == 0
            else int(value) - transient_offset
            if int(value) >= transient_offset
            else int(value)
        )
    )

    events, intervals = [], []
    event_number = 0
    for start, end, active_code in contiguous_runs(event_codes):
        if pd.isna(active_code):
            continue
        active_code = int(active_code)
        segment = labels.iloc[start:end]
        transient = segment.index[segment.eq(transient_offset + active_code)]
        steady = segment.index[segment.eq(active_code)]
        if len(transient) and len(steady) and steady[0] < transient[0]:
            raise ValueError(f"Steady phase precedes transient phase in {relative_path}")
        observable_ts = timestamps.iloc[int(transient[0])] if len(transient) else timestamps.iloc[start]
        impact_ts = timestamps.iloc[int(steady[0])] if len(steady) else pd.NaT
        end_ts = timestamps.iloc[end] if end < len(timestamps) else final_end
        fault_id = f"3W-{instance_id}-{active_code}-{event_number:02d}"
        events.append({
            "fault_id": fault_id,
            "fault_type": definitions[active_code]["description"],
            "domain_id": entity_id,
            "onset_ts": timestamps.iloc[start],
            "observable_ts": observable_ts,
            "impact_ts": impact_ts,
            "end_ts": end_ts,
            "group_id": pd.NA,
            "label_source": "3w_class_phase_derived",
            "source_instance_id": instance_id,
        })
        intervals.append({
            "fault_id": fault_id,
            "entity_id": entity_id,
            "start_ts": timestamps.iloc[start],
            "end_ts": end_ts,
            "label_source": "3w_class_interval",
            "source_instance_id": instance_id,
        })
        event_number += 1
    return events, intervals, conditions


In [ ]:
def build_threew_pack(
    source,
    destination,
    selection,
    partitions,
    *,
    include_evaluation=True,
    batch_rows=50_000,
):
    source, destination = Path(source), Path(destination)
    if batch_rows < 1:
        raise ValueError("batch_rows must be positive")
    if include_evaluation:
        definitions, transient_offset = event_definitions(source)
    else:
        definitions, transient_offset = {}, None
    catalogue = available_threew_catalogue(source, selection)
    if catalogue.empty:
        raise ValueError("No mapped measurements were observed in the selection")
    rename_metrics = dict(zip(catalogue["native_field"], catalogue["metric_id"]))

    with new_output_directory(destination) as pack:
        core = pack / "PACK-CORE"
        observations = core / "observations"
        observations.mkdir(parents=True)
        event_rows, interval_rows, condition_rows = [], [], []
        part_number = 0

        for item in selection.itertuples(index=False):
            relative_path = item.relative_path
            native = read_threew_file(source / relative_path)
            episode_fields = [
                field for field in catalogue["native_field"]
                if field in native.columns and native[field].notna().any()
            ]
            if not episode_fields:
                raise ValueError(f"No observed metrics in {relative_path}")
            episode_metric_ids = [rename_metrics[field] for field in episode_fields]
            for start in range(0, len(native), batch_rows):
                batch = native.iloc[start:start + batch_rows]
                wide = batch[["event_ts", *episode_fields]].rename(columns=rename_metrics)
                wide.insert(1, "entity_id", item.entity_id)
                wide.insert(2, "episode_id", item.source_instance_id)
                long = wide.melt(
                    id_vars=["event_ts", "entity_id", "episode_id"],
                    value_vars=episode_metric_ids,
                    var_name="metric_id",
                    value_name="value",
                )
                long["quality_code"] = "measured"
                long.loc[long["value"].isna(), "quality_code"] = "invalid"
                long = long[PACK_OBSERVATION_SCHEMA]
                long.to_parquet(
                    observations / f"part-{part_number:05d}.parquet",
                    index=False,
                    compression="zstd",
                )
                part_number += 1
            if include_evaluation:
                events, intervals, conditions = threew_truth(
                    native, relative_path, definitions, transient_offset
                )
                event_rows.extend(events)
                interval_rows.extend(intervals)
                condition_rows.extend(conditions)

        catalogue[PACK_METRIC_SCHEMA].to_parquet(
            core / "metric_catalogue.parquet", index=False
        )
        pd.DataFrame({
            "entity_id": sorted(selection["entity_id"].unique()),
            "entity_type": "oil_well",
        })[PACK_ENTITY_SCHEMA].to_parquet(core / "entity_registry.parquet", index=False)
        selection.rename(columns={"source_instance_id": "episode_id"}).assign(
            episode_basis="source_recording"
        )[PACK_EPISODE_SCHEMA].drop_duplicates().to_parquet(
            core / "observation_episodes.parquet", index=False
        )

        splits = pack / "SPLITS"
        splits.mkdir()
        partitions[SPLIT_SCHEMAS["entity_partitions"]].to_parquet(
            splits / "entity_partitions.parquet", index=False
        )

        evaluation_tables = []
        if include_evaluation:
            evaluation = pack / "PACK-EVAL"
            evaluation.mkdir()
            tables = {
                "fault_events": pd.DataFrame(event_rows, columns=EVAL_SCHEMAS["fault_events"]),
                "fault_entity_intervals": pd.DataFrame(interval_rows, columns=EVAL_SCHEMAS["fault_entity_intervals"]),
                "condition_states": pd.DataFrame(condition_rows, columns=EVAL_SCHEMAS["condition_states"]),
            }
            evaluation_tables = list(tables)
            for name, frame in tables.items():
                frame.to_parquet(evaluation / f"{name}.parquet", index=False)

        selected_files = selection["relative_path"].tolist()
        source_files = [
            source_file(source / "dataset.ini", source, "source_metadata"),
            *[
                source_file(source / relative, source, "model_and_evaluation_source")
                for relative in selected_files
            ],
        ]
        save_pack(
            pack,
            sector="petrobras_3w",
            pack_version="0.6.2",
            source_info={
                "source_id": "petrobras-3w-2.0.0",
                "source_root": str(source),
                "files": source_files,
                "selected_files": selected_files,
                "selection_rule": "lowest SHA-256 path key per real well and event class",
                "split_rule": "whole-well 50/25/25 deterministic hash search with minimum evaluation coverage",
                "fields_routed_only_to_evaluation": ["class", "state"],
            },
            evaluation_tables=evaluation_tables,
            split_tables=["entity_partitions"],
            notes=[
                "Real WELL recordings only.",
                "One deterministic file per well and event class by default.",
                "No reporting obligation is inferred between recordings.",
                "Every source recording has a distinct episode_id.",
                "Development and holdout each contain normal and undesirable-event recordings.",
                "Observation presence is explicit at metric level.",
                "All-null metrics are unavailable, not invalid observations.",
                "Partial null values are marked invalid.",
                "Naive source timestamps are normalized to UTC for the canonical contract; no physical source timezone is inferred.",
                "One source recording is loaded at a time; batch_rows bounds only the long-form reshape.",
            ],
        )
    return read_json(destination / "pack_manifest.json")


if RUN_BUILD:
    pack_manifest = build_threew_pack(
        SOURCE,
        PACK_ROOT,
        selected,
        entity_partitions,
        include_evaluation=True,
        batch_rows=BATCH_ROWS,
    )
else:
    pack_manifest = read_json(PACK_ROOT / "pack_manifest.json")

display(pd.Series(pack_manifest["core_row_counts"], name="rows").to_frame())
display(pd.read_parquet(PACK_ROOT / "PACK-CORE" / "observation_episodes.parquet").head())
display(pd.read_parquet(PACK_ROOT / "SPLITS" / "entity_partitions.parquet"))


## 5. Truth-isolation test

The expensive expanded population is not rebuilt for this test. A small,
representative subset is translated before and after `class` and `state`
are removed. `PACK-CORE` must remain identical, and a deliberately leaky
class signature must fail on the redacted source. Asking for evaluation
from the redacted source must also fail explicitly.


In [ ]:
normal = selected.loc[selected["event_code"].eq(0)].head(1)
events = selected.loc[selected["event_code"].gt(0)].drop_duplicates("event_code").head(2)
isolation_selection = pd.concat([normal, events], ignore_index=True)
isolation_partitions = entity_partitions.loc[
    entity_partitions["entity_id"].isin(isolation_selection["entity_id"])
].copy()


def make_redacted_source(source, destination, selection):
    destination.mkdir()
    shutil.copy2(source / "dataset.ini", destination / "dataset.ini")
    for relative_path in selection["relative_path"]:
        target = destination / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        frame = pd.read_parquet(source / relative_path).drop(columns=["class", "state"])
        frame.to_parquet(target)


def deliberately_leaky_signature(source, selection):
    values = []
    for relative_path in selection["relative_path"]:
        path = source / relative_path
        if "class" not in pq.ParquetFile(path).schema_arrow.names:
            return "missing"
        values.append(pd.read_parquet(path, columns=["class"]))
    return str(pd.concat(values)["class"].value_counts().sort_index().to_dict())


with tempfile.TemporaryDirectory() as temporary:
    temporary = Path(temporary)
    redacted_source = temporary / "native_redacted"
    make_redacted_source(SOURCE, redacted_source, isolation_selection)

    original_pack = temporary / "pack_original"
    redacted_pack = temporary / "pack_redacted"
    build_threew_pack(
        SOURCE, original_pack, isolation_selection, isolation_partitions,
        include_evaluation=True, batch_rows=BATCH_ROWS,
    )
    build_threew_pack(
        redacted_source, redacted_pack, isolation_selection, isolation_partitions,
        include_evaluation=False, batch_rows=BATCH_ROWS,
    )

    assert pack_fingerprint(original_pack) == pack_fingerprint(redacted_pack)
    assert deliberately_leaky_signature(SOURCE, isolation_selection) != deliberately_leaky_signature(redacted_source, isolation_selection)

    try:
        build_threew_pack(
            redacted_source, temporary / "pack_invalid",
            isolation_selection, isolation_partitions,
            include_evaluation=True, batch_rows=BATCH_ROWS,
        )
    except ValueError as error:
        assert "Evaluation labels missing" in str(error)
    else:
        raise AssertionError("Evaluation build accepted a source without labels")

print("PASS — PACK-CORE is unchanged after class and state are removed")
print("PASS — the negative control detects the removed labels")
print("PASS — evaluation cannot be requested when labels are absent")


## 6. Contract-fit report and outputs

The report states what the source could not express. That is evidence about
the contract—not a failure to be hidden with fabricated data.


In [ ]:
contract_fit = {
    "represented_cleanly": [
        "multivariate telemetry within event recordings",
        "real well identity",
        "source recording identity as episode_id",
        "metric-level observation presence",
        "all-null episode metrics represented as unavailable",
        "partial source nulls retained for quality coding",
        "class labels physically isolated",
        "condition states represented as intervals",
        "whole-well development partitions",
    ],
    "not_expressible_without_invention": [
        "continuous service obligation between files",
        "shared manifold topology",
        "cross-well cause groups",
        "operator tickets or maintenance actions",
        "independent business-impact timestamps",
    ],
}
display(pd.Series(contract_fit, name="finding").to_frame())
display(pd.Series(read_json(PACK_ROOT / "pack_manifest.json"), name="value").to_frame())
print("Pack root:", PACK_ROOT)
print("Next: 01B_COMMON_CANONICAL_ADAPTER.ipynb")
